# Phase4: アブレーション（拡張の切り分け）と学習枚数の影響

**この回で学ぶこと**
- 拡張を**1種類ずつ**試して、効くもの・効かないものを見分ける方法（アブレーション）
- 学習データ枚数を変えたとき、拡張の効果がどう変わるか（**拡張はデータの代わりにならない**）

> ⏱ 所要時間: QUICK=Trueで約15分（アブレーション5条件×150エポック＋枚数6条件×300エポック、各1シード）。
> 時間がないときは、先に上のセクション1だけ実行してください。

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)

## 1. アブレーション: 拡張を1種類ずつ

| 名前 | 内容 |
|---|---|
| `none` | 拡張なし |
| `flip` | 水平反転のみ |
| `crop` | ランダム切り抜き（scale 0.6〜1.0）のみ |
| `color` | 色調変更のみ |
| `basic` | 切り抜き＋反転＋色調（全部入り） |

「全部入れて良くなった」だけでは、**どれが効いたか**も、**無駄な拡張が混ざっていないか**も分かりません。

In [ ]:
abl = []
for aug in ["none", "flip", "crop", "color", "basic"]:
    print(f"== aug={aug} ==")
    d, _ = run_seeds(SEEDS, aug=aug, epochs=EPOCHS)
    abl.append(d)
abl = pd.concat(abl)
mean_table(abl)

In [ ]:
g = abl.groupby("aug", sort=False)["test_acc"]
order = ["none", "flip", "crop", "color", "basic"]
m, s = g.mean().loc[order], g.std().fillna(0).loc[order]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(order, m, yerr=s, capsize=4, color=["gray", "C0", "C1", "C2", "C3"])
ax.set(ylim=(0.5, 1.0), ylabel="test精度", title="拡張の種類ごとの test 精度（平均±標準偏差）")
ax.grid(axis="y", alpha=0.3);

### 結果の読み方（MANUAL.mdの記載値・3シード平均, test）
拡張なし 0.701 / 水平反転 0.757 / **ランダム切り抜き 0.835** / 色調のみ 0.701 / 全部入り 0.796

- **切り抜きが最も効く**（+13pt）。被写体の位置・大きさのばらつきに強くなる。
- **色調変更だけでは効果なし。** アリ/ハチの判別は形が手がかりで、色の揺らぎは無関係だった可能性。
- 全部入りは過学習を最も抑える（gap最小）が、test は切り抜き単独より低い（ただし誤差範囲）。**全部入りが最良とは限らない。**

## 2. 学習データ枚数を変える

クラスあたり 10 / 30 / 60 枚を無作為に抽出し、拡張なし vs `basic` を比べます（300エポック）。
枚数が少ないほど過学習しやすいので、拡張の効果が大きく出ると期待できますが…？

In [ ]:
sizes = []
for n in [10, 30, 60]:
    for aug in ["none", "basic"]:
        print(f"== n/class={n}, aug={aug} ==")
        d, _ = run_seeds(SEEDS, aug=aug, epochs=ep(300), n_per_class=n)
        sizes.append(d)
sizes = pd.concat(sizes)
mean_table(sizes, by=("n_per_class", "aug"))

In [ ]:
pv = sizes.assign(n=sizes.n_per_class.astype(int)).groupby(["n", "aug"])[["test_acc", "gap"]].mean().unstack()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
pv["test_acc"].plot(marker="o", ax=ax[0], title="test精度")
pv["gap"].plot(marker="o", ax=ax[1], title="過学習の差 (train-val)")
for a in ax:
    a.set_xlabel("クラスあたり学習枚数"); a.grid(alpha=0.3)
fig.tight_layout();

### 結果の読み方（MANUAL.mdの記載値・3シード平均）

| 枚数/クラス | test（なし → あり） | gap（なし → あり） |
|---|---|---|
| 10 | 0.640 → 0.636 | 0.386 → 0.335 |
| 30 | 0.731 → 0.727 | 0.350 → 0.241 |
| 60 | 0.658 → 0.731 | 0.303 → 0.154 |

- gap（過学習）は**どの枚数でも拡張で縮む**。
- しかし **test が改善したのは60枚のみ**。10〜30枚では変わらない。
- → **拡張は「あるデータの見え方のばらつき」を補うもので、新しい情報は作れない。** 極端に少ないときはデータ追加や転移学習（次回）が必要。

## 🧪 試してみよう
1. `T.AUG_SETS["crop"]` の `scale=(0.6, 1.0)` を `(0.2, 1.0)` に変えると、効果は強まる？ 弱まる？
2. 水平反転の代わりに**垂直反転**を `AUG_SETS` に追加して比べよう。アリ/ハチに有効？（向きに意味があるか考える）
3. `crop` + `flip` の2つだけの組み合わせ（`cropflip`）と `crop` 単独を比べよう。
4. 色調変更が効かなかった理由を、画像を見ながら考えよう。もしクラスが「熟したバナナ/未熟なバナナ」なら、色調変更は有効か？

## まとめ
- 拡張は**1種類ずつ**試して効果を切り分ける。無駄・逆効果の拡張に気付ける。
- 効く拡張は**課題によって違う**（形が手がかり→切り抜き、色が手がかり→色調は避ける）。
- 拡張は**データの代わりにならない**。

➡ 次は `05_advanced.ipynb`（RandAugment・Mixup/CutMix・転移学習）